# 11 · Modelo de Dos Etapas — Especialista TWF

## Motivación

El modelo final (notebook 06_RF_Afinado) deja escapar **7 falsos negativos invisibles**:
son todos TWF (Tool Wear Failures) con desgaste alto pero torque moderado.
El modelo principal no los detecta porque su atención está repartida entre los 5 tipos de fallo.

## Arquitectura propuesta

```
ETAPA 1 — Especialista TWF
  Input:  features de desgaste (Tool wear, wear_ratio, Type, Torque, Wear_torque)
  Target: TWF = 1  vs  operación normal = 0
  Output: P(riesgo de rotura de herramienta) para cada observación
                    ↓
ETAPA 2 — Stacking enriquecido
  Input:  features originales (10) + P(TWF_risk) de etapa 1 = 11 features
  Target: Machine failure (cualquier tipo)
  Output: P(fallo de máquina) — con señal TWF más precisa
```

## Clave anti-leakage

El P(TWF_risk) para el conjunto de train se genera con **predicciones out-of-fold**
(cross_val_predict), igual que hace el StackingClassifier internamente.
El modelo de etapa 1 nunca ve los datos de test durante el entrenamiento.


In [ ]:
import sys; sys.path.insert(0, '../src')
import pickle, warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, recall_score, precision_score,
                              roc_auc_score, precision_recall_curve)
from utils import load_ai4i, get_splits, FEATURE_COLS, eval_threshold

np.random.seed(42)

ai4i = load_ai4i('../data/raw/ai4i2020.csv')
X_train, X_test, y_train, y_test, i_train, i_test, _ = get_splits(ai4i)

y_twf       = ai4i['TWF'].values
y_twf_train = y_twf[i_train]
y_twf_test  = y_twf[i_test]

with open('../data/processed/models.pkl', 'rb') as f:
    models = pickle.load(f)

print(f'Train: {X_train.shape} | Fallos: {y_train.sum()} | TWF en train: {y_twf_train.sum()}')
print(f'Test:  {X_test.shape}  | Fallos: {y_test.sum()}  | TWF en test:  {y_twf_test.sum()}')
print(f'Cargados desde utils: {len(FEATURE_COLS)} features')


---
## 1. Etapa 1 — Especialista TWF

### Datos de entrenamiento del especialista
Solo usamos dos clases:
- **Positivo (1):** operaciones que terminaron en TWF
- **Negativo (0):** operaciones completamente normales (sin ningún fallo)

Excluimos deliberadamente los otros fallos (HDF, PWF, OSF, RNF) del entrenamiento
del especialista para que aprenda la frontera precisa entre "desgaste normal" y "rotura".

### Features del especialista
Solo las features físicamente relevantes para el desgaste de herramienta:
`Tool wear`, `wear_ratio`, `Type_enc`, `Torque`, `Wear_torque`.
Cuantas menos features, más foco en la señal TWF.

### Anti-leakage: out-of-fold predictions
Para construir el dataset de entrenamiento de Etapa 2 sin filtración de datos,
generamos el P(TWF_risk) de cada muestra de train usando cross_val_predict:
el modelo NUNCA predice sobre muestras que usó para entrenarse en ese fold.


In [ ]:
# ── Índices de features TWF-relevantes ────────────────────────────────────
TWF_FEATS = ['Type_enc', 'Torque [Nm]', 'Tool wear [min]', 'Wear_torque', 'wear_ratio']
twf_idx   = [feature_cols.index(f) for f in TWF_FEATS]
print(f'Features especialista ({len(twf_idx)}):', TWF_FEATS)

# ── Distribución TWF en train ──────────────────────────────────────────────
n_twf    = y_twf_train.sum()
n_normal = (y_train == 0).sum()
spw_twf  = n_normal / n_twf
print(f'\nTWF en train: {n_twf} | Normales: {n_normal}')
print(f'scale_pos_weight especialista: {spw_twf:.1f}')

# ── Modelo especialista ────────────────────────────────────────────────────
stage1 = LGBMClassifier(
    scale_pos_weight = spw_twf,
    n_estimators     = 300,
    learning_rate    = 0.05,
    num_leaves       = 31,
    random_state     = 42,
    verbose          = -1,
)

# ── Out-of-fold P(TWF_risk) para train (anti-leakage) ─────────────────────
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

p_twf_oof = cross_val_predict(
    stage1,
    X_train[:, twf_idx],
    y_twf_train,
    cv=cv5,
    method='predict_proba',
    n_jobs=-1,
)[:, 1]

print(f'\nP(TWF_risk) OOF — min: {p_twf_oof.min():.3f} | '
      f'max: {p_twf_oof.max():.3f} | '
      f'media en TWF: {p_twf_oof[y_twf_train==1].mean():.3f} | '
      f'media en Normal: {p_twf_oof[y_train==0].mean():.3f}')

# ── Entrenar especialista final (para predecir en test) ───────────────────
stage1.fit(X_train[:, twf_idx], y_twf_train)
p_twf_test = stage1.predict_proba(X_test[:, twf_idx])[:, 1]

print(f'P(TWF_risk) test — media en TWF real: '
      f'{p_twf_test[y_twf_test==1].mean():.3f} | '
      f'media en Normal: {p_twf_test[y_test==0].mean():.3f}')


## 2. Análisis del especialista — ¿separa bien los TWF?

Antes de usar P(TWF_risk) como feature, comprobamos que el especialista
realmente aprende algo: ¿las distribuciones de probabilidad de TWF y No-TWF son distintas?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de P(TWF_risk) en test
ax = axes[0]
for label, name, color in [(0, 'No TWF', '#AED6F1'), (1, 'TWF', '#F1948A')]:
    mask = y_twf_test == label
    ax.hist(p_twf_test[mask], bins=40, alpha=0.7,
            color=color, label=f'{name} (n={mask.sum()})', density=True)
ax.set_xlabel('P(TWF_risk)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución P(TWF_risk) — especialista etapa 1')
ax.legend()

# P(TWF_risk) de los FN del modelo anterior vs TP
y_prob_prev = models['stacking_best'].predict_proba(X_test)[:, 1]
thr_prev    = models['stacking_best_threshold']
y_pred_prev = (y_prob_prev >= thr_prev).astype(int)

fn_mask = (y_test == 1) & (y_pred_prev == 0)   # falsos negativos anteriores
tp_mask = (y_test == 1) & (y_pred_prev == 1)   # verdaderos positivos anteriores

ax = axes[1]
ax.scatter(p_twf_test[tp_mask], y_prob_prev[tp_mask],
           color='seagreen', s=60, label=f'TP ({tp_mask.sum()})', zorder=3)
ax.scatter(p_twf_test[fn_mask], y_prob_prev[fn_mask],
           color='tomato', s=80, marker='X', label=f'FN ({fn_mask.sum()})', zorder=4)
ax.axhline(thr_prev, color='black', ls='--', lw=1, label=f'Umbral anterior ({thr_prev:.2f})')
ax.set_xlabel('P(TWF_risk) — especialista etapa 1')
ax.set_ylabel('P(fallo) — modelo anterior')
ax.set_title('FN del modelo anterior vs P(TWF_risk)')
ax.legend(fontsize=8)

plt.suptitle('Evaluación del especialista TWF', fontsize=13)
plt.tight_layout()
plt.show()

# ¿Cuántos FN tienen P(TWF_risk) alta?
fn_twf_mask = fn_mask & (y_twf_test == 1)
print(f'FN totales: {fn_mask.sum()}')
print(f'FN que son TWF: {fn_twf_mask.sum()}')
print(f'  P(TWF_risk) de esos FN TWF: {p_twf_test[fn_twf_mask].round(3)}')
print(f'  P(fallo) anterior de esos FN: {y_prob_prev[fn_twf_mask].round(3)}')


## 3. Etapa 2 — Stacking enriquecido con P(TWF_risk)

Añadimos P(TWF_risk) como feature número 11 al dataset de entrenamiento.

- **En train:** usamos los P(TWF_risk) out-of-fold (sin leakage)
- **En test:** usamos las predicciones del especialista entrenado sobre todo train

Reentrenamos el mismo StackingClassifier (LGBM + RF afinado) con esta feature extra.


In [ ]:
# ── Feature matrices aumentadas ───────────────────────────────────────────
X_train_aug = np.column_stack([X_train, p_twf_oof])
X_test_aug  = np.column_stack([X_test,  p_twf_test])

print(f'Features originales: {X_train.shape[1]}  →  Con P(TWF_risk): {X_train_aug.shape[1]}')

# ── Mismos base learners del mejor modelo ─────────────────────────────────
lgbm_s2 = LGBMClassifier(
    colsample_bytree=0.7183, learning_rate=0.030,
    min_child_samples=32, n_estimators=127, num_leaves=63,
    reg_alpha=0.4165, reg_lambda=0.8833, subsample=0.6488,
    scale_pos_weight=42.78, random_state=42, verbose=-1
)

rf_params = models['rf_tuned'].get_params()
rf_s2 = RandomForestClassifier(
    **{k: v for k, v in rf_params.items()
       if k in ('n_estimators','max_depth','min_samples_split',
                'min_samples_leaf','max_features','class_weight')},
    random_state=42, n_jobs=-1
)

print('Entrenando Stacking de Etapa 2...')
stack_s2 = StackingClassifier(
    estimators=[('lgbm', lgbm_s2), ('rf', rf_s2)],
    final_estimator=LogisticRegression(class_weight='balanced',
                                       max_iter=1000, random_state=42),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    passthrough=False, n_jobs=-1,
)
stack_s2.fit(X_train_aug, y_train)
print('Listo.')

# ── Umbral óptimo ──────────────────────────────────────────────────────────
def find_threshold(y_true, y_prob, min_recall=0.85):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1  = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
    mask = rec[:-1] >= min_recall
    idx  = np.argmax(f1*mask) if mask.any() else np.argmax(f1)
    return thr[idx], prec[idx], rec[idx], f1[idx]

y_prob_s2 = stack_s2.predict_proba(X_test_aug)[:, 1]
thr_s2, prec_s2, rec_s2, f1_s2 = find_threshold(y_test, y_prob_s2)
y_pred_s2 = (y_prob_s2 >= thr_s2).astype(int)

print(f'\n=== Stacking Etapa 2 — umbral {thr_s2:.4f} ===')
print(classification_report(y_test, y_pred_s2, target_names=['Normal', 'Fallo']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_s2):.4f}')


## 4. Comparativa — modelo anterior vs modelo de dos etapas


In [ ]:
# ── Métricas comparadas ────────────────────────────────────────────────────
prev_f1   = f1_score(y_test, y_pred_prev)
prev_rec  = recall_score(y_test, y_pred_prev)
prev_prec = precision_score(y_test, y_pred_prev)
prev_auc  = roc_auc_score(y_test, y_prob_prev)

comp = pd.DataFrame({
    'Stacking original (1 etapa)': {
        'Precision': round(prev_prec, 4), 'Recall': round(prev_rec, 4),
        'F1': round(prev_f1, 4), 'ROC-AUC': round(prev_auc, 4),
        'Umbral': round(thr_prev, 4), 'FN': fn_mask.sum(),
        'FP': ((y_pred_prev==1)&(y_test==0)).sum()
    },
    'Stacking dos etapas (+P_TWF)': {
        'Precision': round(prec_s2, 4), 'Recall': round(rec_s2, 4),
        'F1': round(f1_s2, 4), 'ROC-AUC': round(roc_auc_score(y_test, y_prob_s2), 4),
        'Umbral': round(thr_s2, 4), 'FN': ((y_pred_s2==0)&(y_test==1)).sum(),
        'FP': ((y_pred_s2==1)&(y_test==0)).sum()
    },
}).T
print(comp.to_string())

# ── Matrices de confusión ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, preds, title in [
    (axes[0], y_pred_prev, f'Original (thr={thr_prev:.2f})'),
    (axes[1], y_pred_s2,   f'Dos etapas (thr={thr_s2:.2f})'),
]:
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Fallo'],
                yticklabels=['Normal', 'Fallo'])
    ax.set_title(title); ax.set_ylabel('Real'); ax.set_xlabel('Predicho')
plt.suptitle('Comparativa matrices de confusión', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 5. ¿Rescatamos algún FN invisible?

La pregunta clave: ¿alguno de los 7 TWF invisibles ahora supera el umbral?


In [ ]:
fn_prev = set(np.where(fn_mask)[0])
fn_s2   = set(np.where((y_pred_s2==0) & (y_test==1))[0])

rescued  = fn_prev - fn_s2          # estaban en FN antes, ahora detectados
new_miss = fn_s2 - fn_prev          # estaban bien, ahora perdidos

print(f'FN modelo original:     {len(fn_prev)}')
print(f'FN modelo dos etapas:   {len(fn_s2)}')
print(f'Fallos RESCATADOS:      {len(rescued)}')
print(f'Fallos nuevos perdidos: {len(new_miss)}')

if rescued:
    print('\n→ Detalles de fallos rescatados:')
    for i in rescued:
        orig_row = idx_test[i]
        print(f'  idx_test={i} | TWF={y_twf_test[i]} | '
              f'Tool wear={ai4i.iloc[orig_row]["Tool wear [min]"]:.0f} min | '
              f'Torque={ai4i.iloc[orig_row]["Torque [Nm]"]:.1f} Nm | '
              f'P_prev={y_prob_prev[i]:.3f} → P_s2={y_prob_s2[i]:.3f}')
else:
    print('\n→ No se rescataron fallos adicionales con el umbral actual.')

# ── Curvas PR comparadas ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for name, yp, color, lw in [
    ('Original (1 etapa)',   y_prob_prev, 'steelblue', 1.5),
    ('Dos etapas (+P_TWF)',  y_prob_s2,  'tomato',    2.5),
]:
    prec_, rec_, _ = precision_recall_curve(y_test, yp)
    ax.plot(rec_, prec_, color=color, lw=lw, label=name)
ax.axvline(0.85, color='black', ls='--', lw=1, label='Recall objetivo')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall — 1 etapa vs 2 etapas')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Importancia de P(TWF_risk) en el modelo final

¿Cuánto peso le da el stacking a la nueva feature frente a las 10 originales?


In [ ]:
feat_names_aug = ['Type', 'Air Temp', 'Process Temp', 'Rot Speed', 'Torque',
                  'Tool Wear', 'Power', 'Temp Diff', 'Wear×Torque', 'wear_ratio',
                  'P(TWF_risk)']   # feature nueva en posición 10

# Importancia del LGBM dentro del stacking
lgbm_fitted = stack_s2.named_estimators_['lgbm']
imp_lgbm = pd.Series(
    lgbm_fitted.booster_.feature_importance(importance_type='gain'),
    index=feat_names_aug
).sort_values(ascending=False)

# Importancia del RF
rf_fitted = stack_s2.named_estimators_['rf']
imp_rf = pd.Series(rf_fitted.feature_importances_,
                   index=feat_names_aug).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, imp, title, color in [
    (axes[0], imp_lgbm, 'LightGBM (gain)', 'crimson'),
    (axes[1], imp_rf,   'Random Forest (Gini)', 'seagreen'),
]:
    imp.sort_values().plot(kind='barh', ax=ax, color=[
        'gold' if i == 'P(TWF_risk)' else color for i in imp.sort_values().index
    ])
    ax.set_title(title)
    ax.set_xlabel('Importancia')

plt.suptitle('Importancia de features — stacking dos etapas\n(P(TWF_risk) en amarillo)',
             fontsize=12)
plt.tight_layout()
plt.show()

rank_lgbm = list(imp_lgbm.index).index('P(TWF_risk)') + 1
rank_rf   = list(imp_rf.index).index('P(TWF_risk)') + 1
print(f'Ranking P(TWF_risk): #{rank_lgbm} en LGBM | #{rank_rf} en RF '
      f'(de {len(feat_names_aug)} features)')


## 7. Guardar modelo de dos etapas


In [ ]:
with open('../data/processed/models.pkl', 'rb') as f:
    models_all = pickle.load(f)

models_all['stage1_twf']            = stage1
models_all['stage2_stacking']       = stack_s2
models_all['stage2_threshold']      = thr_s2
models_all['stage1_twf_feat_idx']   = twf_idx
models_all['stage2_scaler']         = scaler

with open('../data/processed/models.pkl', 'wb') as f:
    pickle.dump(models_all, f)

print('Guardados: stage1_twf, stage2_stacking, stage2_threshold')
print(f'Modelo dos etapas: F1={f1_s2:.4f} | Recall={rec_s2:.4f} | '
      f'AUC={roc_auc_score(y_test, y_prob_s2):.4f}')


## 8. Conclusiones

### Qué aporta la arquitectura de dos etapas

| Aspecto | Modelo original | Modelo dos etapas |
|---|---|---|
| **Arquitectura** | StackingClassifier(LGBM+RF) | Especialista TWF → Stacking enriquecido |
| **Features** | 10 | 10 + P(TWF_risk) = 11 |
| **Especialización** | Aprende los 5 tipos de fallo a la vez | Etapa 1 dedicada solo a TWF |

### Sobre los 7 FN invisibles

Si el modelo no rescata ninguno, confirma el diagnóstico del notebook 10:
los 7 TWF invisibles tienen umbrales de rotura aleatorios e individuales
**no observables en los sensores**. Ni el especialista más afinado puede
detectarlos sin conocer el umbral asignado a cada herramienta.

La solución real requeriría datos que actualmente no existen en el dataset:
- `tool_id` — identificador de herramienta individual
- `ops_since_last_change` — historial de uso de esa herramienta concreta

### Valor de la arquitectura

Aunque la mejora en métricas sea marginal, el modelo de dos etapas demuestra:
1. **Razonamiento arquitectural**: separar un subproblema difícil en una etapa dedicada
2. **Técnica avanzada**: generación out-of-fold para evitar leakage en features derivadas
3. **Diagnóstico honesto**: si no mejora, el análisis explica por qué — y eso es valioso
